# AMEX Enterprise Credit Risk Platform
## Notebook 29 — Phase 2, Problem 4: Delinquency Escalation / Loss Severity — Financial-Impact Reporting & Packaging
### CRISP-DM: Deployment (business handoff)

Fourth and final notebook for Problem 4. Translates Notebooks 26-28's real, validated tier-differentiated
LGD work into a financial-impact package: a Word report, a standalone interactive HTML dashboard, and a
colorful Excel workbook with a real Table + AutoFilter dropdowns + conditional formatting + a linked chart —
plus SMART suggestions for every organizational level from frontline collections to the CFO.

**Honesty boundary:** the tier populations, default rates, and loss dollar amounts below are computed live
from Notebook 27's real holdout scoring. Three financial planning inputs are explicit, editable
`ASSUMPTION`s that this dataset cannot supply: the LGD/EAD figures inherited from Notebook 08, an
early-intervention success rate for the Severe tier, and a one-time implementation cost / rescoring cadence
used only for the ROI and payback-period estimate.

**Deliverables:** `Financial_Impact_Report.docx`, `financial_impact_dashboard.html`,
`AMEX_Problem4_Financial_Impact_Workbook.xlsx`, plus inline charts and tables.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD NOTEBOOK 26/27/28's REAL OUTPUTS
# =============================================================================
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Notebook 26/27/28's Real Outputs")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P1_ROOT = PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
P4_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "Problem4_Delinquency_Escalation_Loss_Severity"
ARTIFACTS_DIR = P4_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

PILLAR_DIRS = {
    "p4_policy": P4_ROOT / "01_LGD_Policy",
    "p4_modeling": P4_ROOT / "02_LGD_Modeling",
    "p4_validation_deployment": P4_ROOT / "03_Validation_Deployment",
    "p4_reporting_packaging": P4_ROOT / "04_Financial_Impact_Reporting_Packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

P1_CONFIG_PATH = P1_ROOT / "artifacts" / "project_config.json"
LGD_POLICY_PATH = PILLAR_DIRS["p4_policy"] / "lgd_policy.json"
TIER_VALIDATION_PATH = PILLAR_DIRS["p4_modeling"] / "tier_validation_summary.json"
NB28_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_28_summary.json"
for _p, _fix in [
    (P1_CONFIG_PATH, "run Problem 1's Notebook 01 first."),
    (LGD_POLICY_PATH, "run Notebook 26 (this problem's Notebook 1) first."),
    (TIER_VALIDATION_PATH, "run Notebook 27 (this problem's Notebook 2) first."),
    (NB28_SUMMARY_PATH, "run Notebook 28 (this problem's Notebook 3) first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p.name} not found in its expected project folder.\nFix: {_fix}")

with open(P1_CONFIG_PATH, "r", encoding="utf-8") as f:
    P1_CONFIG = json.load(f)
with open(LGD_POLICY_PATH, "r", encoding="utf-8") as f:
    LGD_POLICY = json.load(f)
with open(TIER_VALIDATION_PATH, "r", encoding="utf-8") as f:
    TIER_VALIDATION = json.load(f)
with open(NB28_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB28_SUMMARY = json.load(f)

TIER_ORDER = LGD_POLICY["tier_order"]
TIER_STATS = {row["tier"]: row for row in TIER_VALIDATION["tier_stats"]}
EAD_PER_ACCOUNT_USD = LGD_POLICY["ead_per_account_usd"]
# The Moderate tier's LGD is, by Notebook 26's own construction (verified there), identical to
# Notebook 08's real flat baseline -- this IS the "what we used before tiering" comparison point.
BASELINE_LGD = LGD_POLICY["lgd_by_tier"]["values"][1]["lgd"]

N_HOLDOUT = sum(int(TIER_STATS[t]["population"]) for t in TIER_ORDER)
N_DEFAULTERS = sum(int(TIER_STATS[t]["defaulters_in_tier"]) for t in TIER_ORDER)

print(f"Real holdout population (Notebook 27)   : {N_HOLDOUT:,} customers")
print(f"Real observed defaulters                : {N_DEFAULTERS:,}")
print(f"Baseline flat LGD (Notebook 08, real)     : {BASELINE_LGD:.2%}")
print(f"EAD per account (Notebook 08, ASSUMPTION) : ${EAD_PER_ACCOUNT_USD:,}")
print(f"Notebook 28 deployment self-test           : {'PASSED' if NB28_SUMMARY['self_test_passed'] else 'FAILED'}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
except ImportError:
    missing.append("python-docx")
try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.worksheet.table import Table, TableStyleInfo
    from openpyxl.formatting.rule import ColorScaleRule
    from openpyxl.chart import BarChart, Reference
    from openpyxl.utils import get_column_letter
except ImportError:
    missing.append("openpyxl")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: FINANCIAL PLANNING ASSUMPTIONS (EXPLICIT, EDITABLE)
# =============================================================================
_section("SECTION 3: Financial Planning Assumptions (Explicit, Editable)")

# --- This dataset has no real collections-intervention outcomes or project-cost data.
#     Every figure below is a stated, editable ASSUMPTION -- edit to your institution's
#     own numbers. Nothing here is fabricated as if it were measured. ---
FINANCIAL_ASSUMPTIONS = {
    "ead_per_account_usd": {"value": EAD_PER_ACCOUNT_USD, "source": "Notebook 08 (inherited, real value read programmatically)"},
    "baseline_flat_lgd": {"value": BASELINE_LGD, "source": "Notebook 08 (inherited, real value read programmatically)"},
    "severe_tier_intervention_success_rate": {
        "value": 0.15,
        "source": "ASSUMPTION -- illustrative conservative early-intervention efficacy figure for the "
                   "highest-severity tier; edit to your institution's own collections-outcome data.",
    },
    "implementation_cost_usd": {
        "value": 60_000,
        "source": "ASSUMPTION -- illustrative one-time build/validate/deploy cost (data science + risk "
                   "review time); edit to your institution's actual project cost.",
    },
    "annual_application_cycles": {
        "value": 4,
        "source": "ASSUMPTION -- how many times per year this scoring pipeline is re-run against a "
                   "holdout-sized population (illustrative quarterly cadence); edit to your institution's "
                   "actual monitoring frequency.",
    },
}
INTERVENTION_SUCCESS_RATE = FINANCIAL_ASSUMPTIONS["severe_tier_intervention_success_rate"]["value"]
IMPLEMENTATION_COST_USD = FINANCIAL_ASSUMPTIONS["implementation_cost_usd"]["value"]
ANNUAL_APPLICATION_CYCLES = FINANCIAL_ASSUMPTIONS["annual_application_cycles"]["value"]

assumptions_path = ARTIFACTS_DIR / "financial_assumptions.json"
with open(assumptions_path, "w", encoding="utf-8") as f:
    json.dump(FINANCIAL_ASSUMPTIONS, f, indent=2)
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    print(f"  {_k}: {_v['value']}  ({_v['source']})")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: REAL FINANCIAL IMPACT -- FLAT VS TIER-DIFFERENTIATED LGD
# =============================================================================
_section("SECTION 4: Real Financial Impact -- Flat vs Tier-Differentiated LGD")

# --- Realized loss on the REAL holdout defaulters, comparing Notebook 08's flat-LGD
#     approach against Notebook 26/27's tier-differentiated LGD. Population and default
#     counts are real (Notebook 27); only the $/account (EAD, LGD-per-tier) are ASSUMPTION. ---
tier_loss_rows = []
loss_flat_usd = 0.0
loss_tier_usd = 0.0
for _t in TIER_ORDER:
    _row = TIER_STATS[_t]
    _n_def = int(_row["defaulters_in_tier"])
    _lgd = float(_row["lgd_assigned"])
    _loss_flat_t = _n_def * EAD_PER_ACCOUNT_USD * BASELINE_LGD
    _loss_tier_t = _n_def * EAD_PER_ACCOUNT_USD * _lgd
    loss_flat_usd += _loss_flat_t
    loss_tier_usd += _loss_tier_t
    tier_loss_rows.append({
        "tier": _t, "population": int(_row["population"]), "defaulters": _n_def,
        "observed_default_rate": float(_row["observed_default_rate"]), "lgd_assigned": _lgd,
        "loss_flat_lgd_usd": round(_loss_flat_t, 2), "loss_tier_lgd_usd": round(_loss_tier_t, 2),
    })
tier_loss_df = pd.DataFrame(tier_loss_rows)

LOSS_DELTA_USD = loss_tier_usd - loss_flat_usd
_direction = "MORE" if LOSS_DELTA_USD > 0 else "LESS"
_interpretation = (
    "the flat 45% LGD was UNDER-recognizing loss on this real portfolio -- tiering surfaces "
    "regulatory/under-provisioning risk the flat model was hiding."
    if LOSS_DELTA_USD > 0 else
    "the flat 45% LGD was OVER-recognizing loss on this real portfolio -- tiering frees "
    "capital that was being conservatively over-reserved."
)
print(tier_loss_df.to_string(index=False))
print(f"\nTotal loss under flat LGD (Notebook 08 approach)      : ${loss_flat_usd:,.0f}")
print(f"Total loss under tier-differentiated LGD (this problem): ${loss_tier_usd:,.0f}")
print(f"Difference                                             : ${LOSS_DELTA_USD:,.0f} ({_direction} recognized)")
print(f"Interpretation: {_interpretation}")
print(f"(Measured on this real holdout of {N_HOLDOUT:,} customers -- scale by your own "
      f"total-portfolio-size ratio for an institution-wide figure.)")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: LOSS-PREVENTION OPPORTUNITY -- EARLY INTERVENTION ON THE SEVERE TIER
# =============================================================================
_section("SECTION 5: Loss-Prevention Opportunity -- Early Intervention on the Severe Tier")

_severe = TIER_STATS[TIER_ORDER[-1]]
_severe_defaulters = int(_severe["defaulters_in_tier"])
_severe_lgd = float(_severe["lgd_assigned"])
PREVENTABLE_DEFAULTS = round(_severe_defaulters * INTERVENTION_SUCCESS_RATE)
LOSS_PREVENTED_USD = PREVENTABLE_DEFAULTS * EAD_PER_ACCOUNT_USD * _severe_lgd

print(f"Severe-tier real defaulters (holdout)         : {_severe_defaulters:,}")
print(f"ASSUMPTION intervention success rate           : {INTERVENTION_SUCCESS_RATE:.0%}")
print(f"Estimated preventable defaults                 : {PREVENTABLE_DEFAULTS:,}")
print(f"Estimated loss prevented (this holdout sample) : ${LOSS_PREVENTED_USD:,.0f}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: ROI, INVESTMENT & PAYBACK PERIOD
# =============================================================================
_section("SECTION 6: ROI, Investment & Payback Period")

ANNUAL_BENEFIT_USD = LOSS_PREVENTED_USD * ANNUAL_APPLICATION_CYCLES
ROI_PCT = ((ANNUAL_BENEFIT_USD - IMPLEMENTATION_COST_USD) / IMPLEMENTATION_COST_USD) * 100 if IMPLEMENTATION_COST_USD else None
PAYBACK_MONTHS = (IMPLEMENTATION_COST_USD / (ANNUAL_BENEFIT_USD / 12)) if ANNUAL_BENEFIT_USD > 0 else None

print(f"Amount invested (ASSUMPTION, one-time)         : ${IMPLEMENTATION_COST_USD:,.0f}")
print(f"Estimated annual loss-prevention benefit        : ${ANNUAL_BENEFIT_USD:,.0f} "
      f"(= per-cycle benefit x {ANNUAL_APPLICATION_CYCLES} cycles/year, ASSUMPTION)")
print(f"Estimated ROI (Year 1)                          : {ROI_PCT:,.0f}%" if ROI_PCT is not None else "n/a")
print(f"Estimated payback period                        : {PAYBACK_MONTHS:.1f} months" if PAYBACK_MONTHS else "n/a")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: SMART SUGGESTIONS -- BOTTOM TO TOP MANAGEMENT
# =============================================================================
_section("SECTION 7: SMART Suggestions -- Bottom to Top Management")

SMART_SUGGESTIONS = [
    {"org_level": "Collections Agents (Frontline)",
     "suggestion": f"Prioritize outbound contact on the {_severe_defaulters:,} real Severe-tier accounts "
                   f"first each day -- their observed default rate ({_severe['observed_default_rate']:.1%}) "
                   f"is the highest of any tier. Target: attempt contact within 24 hours of a Severe-tier flag."},
    {"org_level": "Collections Team Lead / Manager",
     "suggestion": f"Track the {PREVENTABLE_DEFAULTS:,}-account early-intervention goal (from the "
                   f"{INTERVENTION_SUCCESS_RATE:.0%} ASSUMPTION success rate) as a weekly team KPI; report "
                   f"actual prevented-default count vs. this target each month to calibrate the assumption."},
    {"org_level": "Risk / Credit Analyst",
     "suggestion": f"Re-run Notebook 27's severity scoring each time Notebook 04's engineered features "
                   f"refresh; monitor the split-half PSI (last measured {NB28_SUMMARY['psi']:.4f}, "
                   f"'{NB28_SUMMARY['psi_verdict']}') and re-tier if it exceeds 0.10 for two consecutive cycles."},
    {"org_level": "Model Risk / Compliance (SR 11-7)",
     "suggestion": f"File this notebook's chi-square (p={NB28_SUMMARY['chi_square_p_value']:.1e}) and "
                   f"z-test (p={NB28_SUMMARY['z_test_p_value']:.1e}) results with the model's annual "
                   f"validation packet; re-certify before any change to the frozen severity_scoring_bundle.json."},
    {"org_level": "Finance / Provisioning Team",
     "suggestion": f"Update the quarterly loss-provisioning workbook to use tier-differentiated LGD "
                   f"instead of the flat {BASELINE_LGD:.0%} -- a ${abs(LOSS_DELTA_USD):,.0f} recognition "
                   f"difference was measured on this holdout; reconcile within the next provisioning cycle."},
    {"org_level": "CFO / Executive Leadership",
     "suggestion": f"Approve the ${IMPLEMENTATION_COST_USD:,.0f} investment given an estimated "
                   f"{PAYBACK_MONTHS:.1f}-month payback and {ROI_PCT:,.0f}% Year-1 ROI from Severe-tier "
                   f"loss prevention alone; revisit the {ANNUAL_APPLICATION_CYCLES}x/year cadence ASSUMPTION "
                   f"at the next quarterly business review."},
]
smart_df = pd.DataFrame(SMART_SUGGESTIONS)
smart_path = PILLAR_DIRS["p4_reporting_packaging"] / "p4_smart_suggestions.csv"
smart_df.to_csv(smart_path, index=False)
for _row in SMART_SUGGESTIONS:
    print(f"[{_row['org_level']}]\n  {_row['suggestion']}\n")
print(f"\u2705 Saved -> {smart_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: INLINE CHARTS
# =============================================================================
_section("SECTION 8: Inline Charts")

VIZ = {"ink": "#0B1F3A", "accent": "#C41E3A", "muted": "#8A93A6", "gold": "#C9A227", "surface": "#FFFFFF"}

fig1, ax1 = plt.subplots(figsize=(7, 5), dpi=150)
_bars = ax1.bar(["Flat LGD\n(Notebook 08)", "Tier-Differentiated LGD\n(this problem)"],
                 [loss_flat_usd, loss_tier_usd], color=[VIZ["muted"], VIZ["accent"]])
for _b, _v in zip(_bars, [loss_flat_usd, loss_tier_usd]):
    ax1.text(_b.get_x() + _b.get_width() / 2, _v, f"${_v:,.0f}", ha="center", va="bottom", fontsize=10)
ax1.set_ylabel("Recognized loss, real holdout defaulters (USD)")
ax1.set_title("Problem 4: Flat vs Tier-Differentiated Loss Recognition")
fig1.tight_layout()
chart1_path = PILLAR_DIRS["p4_reporting_packaging"] / "flat_vs_tier_loss_chart.png"
fig1.savefig(chart1_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig1)

fig2, ax2 = plt.subplots(figsize=(7.5, 5), dpi=150)
_x = np.arange(len(TIER_ORDER))
ax2.bar(_x - 0.2, tier_loss_df["loss_flat_lgd_usd"], width=0.4, label="Flat LGD", color=VIZ["muted"])
ax2.bar(_x + 0.2, tier_loss_df["loss_tier_lgd_usd"], width=0.4, label="Tier LGD", color=VIZ["accent"])
ax2.set_xticks(_x); ax2.set_xticklabels(TIER_ORDER)
ax2.set_ylabel("Recognized loss (USD)")
ax2.set_title("Problem 4: Loss Recognition by Severity Tier")
ax2.legend()
fig2.tight_layout()
chart2_path = PILLAR_DIRS["p4_reporting_packaging"] / "tier_loss_breakdown_chart.png"
fig2.savefig(chart2_path, dpi=150, facecolor=VIZ["surface"])
plt.show()
plt.close(fig2)

print(f"\u2705 Saved -> {chart1_path.name}, {chart2_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: WORD REPORT -- Financial_Impact_Report.docx
# =============================================================================
_section("SECTION 9: Word Report -- Financial_Impact_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = "" if v is None else str(v)
    return table


def _add_table_from_df(doc, df, max_rows=30):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    return table


doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 2, Problem 4: Delinquency Escalation / Loss Severity -- Financial Impact Report")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(doc, "1. Executive Summary", level=1)
doc.add_paragraph(
    f"Tier-differentiated LGD, computed from a real Escalation Severity Score, was validated against "
    f"real observed default rates (Notebooks 26-28) and applied to {N_HOLDOUT:,} real holdout customers "
    f"({N_DEFAULTERS:,} real defaulters). Compared to Notebook 08's flat {BASELINE_LGD:.0%} LGD, this "
    f"changes recognized loss by ${LOSS_DELTA_USD:,.0f} -- {_interpretation} Early intervention on the "
    f"Severe tier (an ASSUMPTION {INTERVENTION_SUCCESS_RATE:.0%} success rate) is estimated to prevent "
    f"${LOSS_PREVENTED_USD:,.0f} of loss per scoring cycle, for an estimated {PAYBACK_MONTHS:.1f}-month "
    f"payback on a ${IMPLEMENTATION_COST_USD:,.0f} implementation investment."
)

_add_heading(doc, "2. Loss Recognition: Flat vs Tier-Differentiated LGD", level=1)
_add_table_from_df(doc, tier_loss_df)
doc.add_paragraph(f"Total flat-LGD loss: ${loss_flat_usd:,.0f}  |  Total tier-LGD loss: ${loss_tier_usd:,.0f}  "
                   f"|  Difference: ${LOSS_DELTA_USD:,.0f}")

_add_heading(doc, "3. Loss-Prevention Opportunity (Severe Tier)", level=1)
_add_kv_table(doc, {"severe_tier_defaulters": _severe_defaulters,
                     "intervention_success_rate_assumption": f"{INTERVENTION_SUCCESS_RATE:.0%}",
                     "preventable_defaults": PREVENTABLE_DEFAULTS,
                     "loss_prevented_per_cycle_usd": f"${LOSS_PREVENTED_USD:,.0f}"})

_add_heading(doc, "4. ROI, Investment & Payback", level=1)
_add_kv_table(doc, {"amount_invested_usd": f"${IMPLEMENTATION_COST_USD:,.0f}",
                     "annual_benefit_usd": f"${ANNUAL_BENEFIT_USD:,.0f}",
                     "roi_year_1_pct": f"{ROI_PCT:,.0f}%", "payback_period_months": f"{PAYBACK_MONTHS:.1f}"})

_add_heading(doc, "5. SMART Suggestions by Organizational Level", level=1)
_add_table_from_df(doc, smart_df)

_add_heading(doc, "6. Assumptions & Sources", level=1)
_assump_df = pd.DataFrame([{"assumption": k, "value": v["value"], "source": v["source"]}
                            for k, v in FINANCIAL_ASSUMPTIONS.items()])
_add_table_from_df(doc, _assump_df)

report_path = PILLAR_DIRS["p4_reporting_packaging"] / "Financial_Impact_Report.docx"
doc.save(str(report_path))
print(f"\u2705 Saved -> {report_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: EXCEL WORKBOOK -- COLORFUL, TABLE + AUTOFILTER + CONDITIONAL FORMATTING + CHART
# =============================================================================
_section("SECTION 10: Excel Workbook -- Colorful, Table + AutoFilter + Conditional Formatting + Chart")

INK = "0B1F3A"
ACCENT = "C41E3A"
GOLD = "C9A227"
LIGHT = "F2F4F8"
WHITE = "FFFFFF"
USD_FMT = '$#,##0;($#,##0);-'

# Deterministic row layout, computed up front so sheets can cross-reference each other by
# formula regardless of the order their cells get written below.
_tier_first_row, _tier_last_row = 2, 1 + len(TIER_ORDER)
_severe_row = _tier_last_row  # Severe is always TIER_ORDER's last element -> last data row
_assump_rows = {k: 2 + i for i, k in enumerate(FINANCIAL_ASSUMPTIONS.keys())}  # e.g. ead_per_account_usd -> row 2

wb = openpyxl.Workbook()

# --- Sheet 1: Assumptions (built first -- every formula below references these cells;
#     yellow-highlighted per financial-model convention, values are the only hardcoded inputs) ---
ws_assump = wb.active
ws_assump.title = "Assumptions"
ws_assump.append(["Assumption", "Value", "Source / Rationale"])
for _k, _v in FINANCIAL_ASSUMPTIONS.items():
    ws_assump.append([_k.replace("_", " ").title(), _v["value"], _v["source"]])
for _r in range(2, ws_assump.max_row + 1):
    ws_assump[f"B{_r}"].fill = PatternFill("solid", fgColor="FFFF00")
    ws_assump[f"B{_r}"].font = Font(name="Calibri", color="0000FF")  # blue = hardcoded input, per convention
    ws_assump[f"C{_r}"].alignment = Alignment(wrap_text=True, vertical="top")
ws_assump[f"B{_assump_rows['baseline_flat_lgd']}"].number_format = "0.0%"
ws_assump[f"B{_assump_rows['severe_tier_intervention_success_rate']}"].number_format = "0.0%"
ws_assump[f"B{_assump_rows['ead_per_account_usd']}"].number_format = USD_FMT
ws_assump[f"B{_assump_rows['implementation_cost_usd']}"].number_format = USD_FMT
ws_assump.column_dimensions["A"].width = 32
ws_assump.column_dimensions["B"].width = 14
ws_assump.column_dimensions["C"].width = 90
_tbl_assump = Table(displayName="Assumptions", ref=f"A1:C{ws_assump.max_row}")
_tbl_assump.tableStyleInfo = TableStyleInfo(name="TableStyleMedium4", showRowStripes=True)
ws_assump.add_table(_tbl_assump)

_ead_ref = f"Assumptions!$B${_assump_rows['ead_per_account_usd']}"
_baseline_lgd_ref = f"Assumptions!$B${_assump_rows['baseline_flat_lgd']}"
_intervention_rate_ref = f"Assumptions!$B${_assump_rows['severe_tier_intervention_success_rate']}"
_cost_ref = f"Assumptions!$B${_assump_rows['implementation_cost_usd']}"
_cycles_ref = f"Assumptions!$B${_assump_rows['annual_application_cycles']}"

# --- Sheet 2: Tier Financial Detail -- Loss columns are REAL FORMULAS (defaulters x EAD x LGD),
#     referencing the Assumptions sheet, so editing an assumption recalculates every loss figure. ---
ws_tier = wb.create_sheet("Tier Financial Detail")
ws_tier.append(["Tier", "Population", "Defaulters", "Observed Default Rate", "LGD Assigned",
                 "Loss -- Flat LGD (USD)", "Loss -- Tier LGD (USD)"])
for _i, _r in tier_loss_df.iterrows():
    _row_n = _tier_first_row + _i
    ws_tier.append([_r["tier"], int(_r["population"]), int(_r["defaulters"]), float(_r["observed_default_rate"]),
                     float(_r["lgd_assigned"]), None, None])
    ws_tier[f"F{_row_n}"] = f"=C{_row_n}*{_ead_ref}*{_baseline_lgd_ref}"
    ws_tier[f"G{_row_n}"] = f"=C{_row_n}*{_ead_ref}*E{_row_n}"
for _col_letter, _fmt in [("D", "0.0%"), ("E", "0.0%"), ("F", USD_FMT), ("G", USD_FMT)]:
    for _r in range(_tier_first_row, _tier_last_row + 1):
        ws_tier[f"{_col_letter}{_r}"].number_format = _fmt
_tbl_tier = Table(displayName="TierFinancialDetail", ref=f"A1:G{_tier_last_row}")
_tbl_tier.tableStyleInfo = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
ws_tier.add_table(_tbl_tier)
ws_tier.conditional_formatting.add(
    f"D{_tier_first_row}:D{_tier_last_row}",
    ColorScaleRule(start_type="min", start_color="63BE7B", end_type="max", end_color="F8696B"))
ws_tier.conditional_formatting.add(
    f"G{_tier_first_row}:G{_tier_last_row}",
    ColorScaleRule(start_type="min", start_color="63BE7B", end_type="max", end_color="F8696B"))
for _col, _w in zip("ABCDEFG", [20, 12, 12, 20, 12, 20, 20]):
    ws_tier.column_dimensions[_col].width = _w

_chart = BarChart()
_chart.title = "Loss Recognition by Severity Tier (filter the table to update this chart)"
_chart.y_axis.title = "Loss (USD)"
_data = Reference(ws_tier, min_col=6, max_col=7, min_row=1, max_row=_tier_last_row)
_cats = Reference(ws_tier, min_col=1, min_row=_tier_first_row, max_row=_tier_last_row)
_chart.add_data(_data, titles_from_data=True)
_chart.set_categories(_cats)
_chart.width, _chart.height = 18, 10
ws_tier.add_chart(_chart, "I2")

# --- Sheet 3: SMART Suggestions (real Excel Table -> native AutoFilter dropdowns) ---
ws_smart = wb.create_sheet("SMART Suggestions")
ws_smart.append(["Org Level", "Suggestion"])
for _row_data in SMART_SUGGESTIONS:
    ws_smart.append([_row_data["org_level"], _row_data["suggestion"]])
_last_row_smart = ws_smart.max_row
_tbl_smart = Table(displayName="SmartSuggestions", ref=f"A1:B{_last_row_smart}")
_tbl_smart.tableStyleInfo = TableStyleInfo(name="TableStyleMedium7", showRowStripes=True)
ws_smart.add_table(_tbl_smart)
ws_smart.column_dimensions["A"].width = 30
ws_smart.column_dimensions["B"].width = 100
for _r in range(2, _last_row_smart + 1):
    ws_smart[f"B{_r}"].alignment = Alignment(wrap_text=True, vertical="top")

# --- Sheet 4: Executive Summary (KPI cards) -- built last so every formula below can reference
#     the Tier Financial Detail and Assumptions sheets that already exist by this point. ---
ws_exec = wb.create_sheet("Executive Summary", 0)  # inserted first, but populated last
wb.active = 0
ws_exec.sheet_view.showGridLines = False
ws_exec["B2"] = "AMEX RiskIQ -- Problem 4: Delinquency Escalation / Loss Severity"
ws_exec["B2"].font = Font(name="Calibri", size=16, bold=True, color=WHITE)
ws_exec["B2"].fill = PatternFill("solid", fgColor=INK)
ws_exec.merge_cells("B2:F2")
ws_exec["B3"] = "Financial Impact Summary"
ws_exec["B3"].font = Font(name="Calibri", size=11, italic=True, color=INK)
ws_exec.merge_cells("B3:F3")

# Rows 5-7 are real formulas (SUM over the Tier Financial Detail table); rows 8-11 are reported
# figures computed from the ASSUMPTION-driven loss-prevention/ROI model in Sections 5-6 above --
# labeled as such below rather than presented as if every cell were a live formula.
_kpi_rows = [
    ("Total Loss -- Flat LGD", f"=SUM('Tier Financial Detail'!F{_tier_first_row}:F{_tier_last_row})", True, LIGHT),
    ("Total Loss -- Tier LGD", f"=SUM('Tier Financial Detail'!G{_tier_first_row}:G{_tier_last_row})", True, LIGHT),
    ("Loss Recognition Change", "=D6-D5", True, GOLD),
    ("Est. Loss Prevented / Cycle", f"${LOSS_PREVENTED_USD:,.0f}  (reported, see Section 5)", False, ACCENT),
    ("Amount Invested", f"=\"$\"&TEXT({_cost_ref},\"#,##0\")", True, LIGHT),
    ("Estimated Year-1 ROI", f"{ROI_PCT:,.0f}%  (reported, see Section 6)", False, ACCENT),
    ("Estimated Payback", f"{PAYBACK_MONTHS:.1f} months  (reported, see Section 6)", False, ACCENT),
]
_row = 5
for _label, _value, _is_formula, _fill in _kpi_rows:
    ws_exec.cell(row=_row, column=2, value=_label).font = Font(name="Calibri", size=11, color=INK)
    _cell = ws_exec.cell(row=_row, column=4, value=_value)
    _cell.font = Font(name="Calibri", size=13, bold=True, color=(WHITE if _fill in (GOLD, ACCENT) else INK))
    _cell.fill = PatternFill("solid", fgColor=_fill)
    _cell.alignment = Alignment(horizontal="center", wrap_text=not _is_formula)
    if _is_formula and _label != "Amount Invested":
        _cell.number_format = USD_FMT
    ws_exec.merge_cells(start_row=_row, start_column=4, end_row=_row, end_column=5)
    _row += 1
ws_exec["B13"] = "Rows 5-7 and 9 recalculate live from the Assumptions and Tier Financial Detail sheets."
ws_exec["B13"].font = Font(name="Calibri", size=9, italic=True, color="8A93A6")
ws_exec.merge_cells("B13:F13")
for _col, _w in zip("BCDEF", [30, 3, 22, 22, 3]):
    ws_exec.column_dimensions[_col].width = _w

for _ws in (ws_tier, ws_smart, ws_assump):
    for _cell in _ws[1]:
        _cell.font = Font(name="Calibri", bold=True, color=WHITE)
        _cell.fill = PatternFill("solid", fgColor=INK)

workbook_path = PILLAR_DIRS["p4_reporting_packaging"] / "AMEX_Problem4_Financial_Impact_Workbook.xlsx"
wb.save(str(workbook_path))
print(f"\u2705 Saved -> {workbook_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: INTERACTIVE HTML DASHBOARD -- DYNAMIC TIER FILTER, LINKED CHART + TABLES
# =============================================================================
_section("SECTION 11: Interactive HTML Dashboard -- Dynamic Tier Filter, Linked Chart + Tables")

_tier_json = json.dumps(tier_loss_rows)
_smart_json = json.dumps(SMART_SUGGESTIONS)
_org_levels = sorted({r["org_level"] for r in SMART_SUGGESTIONS})

_html = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Problem 4 -- Financial Impact Dashboard</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4"></script>
<style>
  :root { --ink:#0B1F3A; --accent:#C41E3A; --gold:#C9A227; --muted:#8A93A6; --bg:#F2F4F8; --card:#FFFFFF; }
  body { font-family: Calibri, Arial, sans-serif; background: var(--bg); color: var(--ink); margin: 0; padding: 24px; }
  h1 { font-size: 22px; margin-bottom: 4px; }
  .sub { color: var(--muted); margin-bottom: 20px; }
  .kpi-row { display: flex; flex-wrap: wrap; gap: 14px; margin-bottom: 24px; }
  .kpi { background: var(--card); border-radius: 10px; padding: 16px 20px; box-shadow: 0 1px 3px rgba(0,0,0,.12); min-width: 190px; flex: 1; }
  .kpi .label { font-size: 12px; color: var(--muted); text-transform: uppercase; }
  .kpi .value { font-size: 22px; font-weight: 700; margin-top: 4px; }
  .panel { background: var(--card); border-radius: 10px; padding: 18px; margin-bottom: 20px; box-shadow: 0 1px 3px rgba(0,0,0,.12); }
  .filters { display: flex; gap: 8px; flex-wrap: wrap; margin-bottom: 12px; }
  .filters button { border: 1px solid var(--ink); background: var(--card); color: var(--ink); padding: 6px 14px; border-radius: 18px; cursor: pointer; font-size: 13px; }
  .filters button.active { background: var(--ink); color: #fff; }
  table { width: 100%; border-collapse: collapse; font-size: 13px; }
  th, td { text-align: left; padding: 8px 10px; border-bottom: 1px solid #E4E7EE; }
  th { background: var(--ink); color: #fff; }
  select { padding: 6px 10px; border-radius: 6px; border: 1px solid var(--muted); font-size: 13px; margin-bottom: 12px; }
  canvas { max-height: 360px; }
</style>
</head>
<body>
<h1>AMEX RiskIQ -- Problem 4: Delinquency Escalation / Loss Severity</h1>
<div class="sub">Financial Impact Dashboard -- real Notebook 27/28 results, ASSUMPTION values clearly marked</div>

<div class="kpi-row">
  <div class="kpi"><div class="label">Total Loss -- Flat LGD</div><div class="value">__LOSS_FLAT__</div></div>
  <div class="kpi"><div class="label">Total Loss -- Tier LGD</div><div class="value">__LOSS_TIER__</div></div>
  <div class="kpi"><div class="label">Loss Recognition Change</div><div class="value">__LOSS_DELTA__</div></div>
  <div class="kpi"><div class="label">Est. Loss Prevented / Cycle</div><div class="value">__LOSS_PREVENTED__</div></div>
  <div class="kpi"><div class="label">Est. Year-1 ROI</div><div class="value">__ROI__</div></div>
  <div class="kpi"><div class="label">Est. Payback</div><div class="value">__PAYBACK__</div></div>
</div>

<div class="panel">
  <div class="filters" id="tierFilters"></div>
  <canvas id="tierChart"></canvas>
  <table id="tierTable"><thead><tr><th>Tier</th><th>Population</th><th>Defaulters</th>
    <th>Default Rate</th><th>LGD</th><th>Loss -- Flat</th><th>Loss -- Tier</th></tr></thead>
    <tbody></tbody></table>
</div>

<div class="panel">
  <label for="orgFilter"><b>SMART Suggestions -- filter by organizational level</b></label><br/>
  <select id="orgFilter"></select>
  <table id="smartTable"><thead><tr><th>Org Level</th><th>Suggestion</th></tr></thead><tbody></tbody></table>
</div>

<script>
const tierData = __TIER_JSON__;
const smartData = __SMART_JSON__;
const orgLevels = __ORG_LEVELS__;
let activeTiers = new Set(tierData.map(d => d.tier));

const fmtUsd = v => "$" + Math.round(v).toLocaleString();
const fmtPct = v => (v * 100).toFixed(1) + "%";

const ctx = document.getElementById("tierChart").getContext("2d");
const chart = new Chart(ctx, {
  type: "bar",
  data: {
    labels: tierData.map(d => d.tier),
    datasets: [
      { label: "Flat LGD Loss", data: tierData.map(d => d.loss_flat_lgd_usd), backgroundColor: "#8A93A6" },
      { label: "Tier LGD Loss", data: tierData.map(d => d.loss_tier_lgd_usd), backgroundColor: "#C41E3A" },
    ],
  },
  options: { responsive: true, plugins: { legend: { position: "top" } },
             scales: { y: { ticks: { callback: v => "$" + v.toLocaleString() } } } },
});

function renderFilters() {
  const box = document.getElementById("tierFilters");
  box.innerHTML = "";
  tierData.forEach(d => {
    const btn = document.createElement("button");
    btn.textContent = d.tier;
    btn.className = activeTiers.has(d.tier) ? "active" : "";
    btn.onclick = () => {
      if (activeTiers.has(d.tier)) { activeTiers.delete(d.tier); } else { activeTiers.add(d.tier); }
      renderFilters(); renderChart(); renderTable();
    };
    box.appendChild(btn);
  });
}

function renderChart() {
  const visible = tierData.filter(d => activeTiers.has(d.tier));
  chart.data.labels = visible.map(d => d.tier);
  chart.data.datasets[0].data = visible.map(d => d.loss_flat_lgd_usd);
  chart.data.datasets[1].data = visible.map(d => d.loss_tier_lgd_usd);
  chart.update();
}

function renderTable() {
  const tbody = document.querySelector("#tierTable tbody");
  tbody.innerHTML = "";
  tierData.filter(d => activeTiers.has(d.tier)).forEach(d => {
    const tr = document.createElement("tr");
    tr.innerHTML = `<td>${d.tier}</td><td>${d.population.toLocaleString()}</td>` +
      `<td>${d.defaulters.toLocaleString()}</td><td>${fmtPct(d.observed_default_rate)}</td>` +
      `<td>${fmtPct(d.lgd_assigned)}</td><td>${fmtUsd(d.loss_flat_lgd_usd)}</td>` +
      `<td>${fmtUsd(d.loss_tier_lgd_usd)}</td>`;
    tbody.appendChild(tr);
  });
}

function renderSmart(filterLevel) {
  const tbody = document.querySelector("#smartTable tbody");
  tbody.innerHTML = "";
  smartData.filter(r => filterLevel === "All" || r.org_level === filterLevel).forEach(r => {
    const tr = document.createElement("tr");
    tr.innerHTML = `<td>${r.org_level}</td><td>${r.suggestion}</td>`;
    tbody.appendChild(tr);
  });
}

const orgSelect = document.getElementById("orgFilter");
["All", ...orgLevels].forEach(level => {
  const opt = document.createElement("option");
  opt.value = level; opt.textContent = level;
  orgSelect.appendChild(opt);
});
orgSelect.onchange = () => renderSmart(orgSelect.value);

renderFilters(); renderChart(); renderTable(); renderSmart("All");
</script>
</body>
</html>
"""
_html = (_html
         .replace("__LOSS_FLAT__", f"${loss_flat_usd:,.0f}")
         .replace("__LOSS_TIER__", f"${loss_tier_usd:,.0f}")
         .replace("__LOSS_DELTA__", f"${LOSS_DELTA_USD:,.0f}")
         .replace("__LOSS_PREVENTED__", f"${LOSS_PREVENTED_USD:,.0f}")
         .replace("__ROI__", f"{ROI_PCT:,.0f}%")
         .replace("__PAYBACK__", f"{PAYBACK_MONTHS:.1f} mo")
         .replace("__TIER_JSON__", _tier_json)
         .replace("__SMART_JSON__", _smart_json)
         .replace("__ORG_LEVELS__", json.dumps(_org_levels)))

dashboard_path = PILLAR_DIRS["p4_reporting_packaging"] / "financial_impact_dashboard.html"
with open(dashboard_path, "w", encoding="utf-8") as f:
    f.write(_html)
print(f"\u2705 Saved -> {dashboard_path.name} (this problem's 04_Financial_Impact_Reporting_Packaging folder)")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: VERIFICATION
# =============================================================================
_section("SECTION 12: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Tier loss table covers all tiers", len(tier_loss_df) == len(TIER_ORDER))
_check("Flat-LGD loss recomputes consistently",
       abs(loss_flat_usd - (N_DEFAULTERS * EAD_PER_ACCOUNT_USD * BASELINE_LGD)) < 1.0)
_check("Loss delta equals tier loss minus flat loss",
       abs(LOSS_DELTA_USD - (loss_tier_usd - loss_flat_usd)) < 1e-6)
_check("ROI/payback are finite, positive numbers", ROI_PCT is not None and PAYBACK_MONTHS is not None
       and PAYBACK_MONTHS > 0)

_expected_files = [assumptions_path, smart_path, chart1_path, chart2_path, report_path,
                    workbook_path, dashboard_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 29 verification checks failed. See \u274c line above.")
print("\nAll Notebook 29 checks passed.")
print("\n\u2705 Section 12 complete.")


# =============================================================================
# SECTION 13: WRITE NOTEBOOK 29 SUMMARY -- PROBLEM 4 COMPLETE
# =============================================================================
_section("SECTION 13: Write Notebook 29 Summary -- Problem 4 Complete")

notebook_29_summary = {
    "notebook": "29_financial_impact_reporting_packaging", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 4, "problem_name": "Delinquency Escalation / Loss Severity",
    "phase": "Phase 2 -- Regulatory & Loss Provisioning", "problem_4_complete": True,
    "loss_flat_lgd_usd": round(loss_flat_usd, 2), "loss_tier_lgd_usd": round(loss_tier_usd, 2),
    "loss_delta_usd": round(LOSS_DELTA_USD, 2), "loss_prevented_per_cycle_usd": round(LOSS_PREVENTED_USD, 2),
    "roi_year_1_pct": round(ROI_PCT, 1), "payback_period_months": round(PAYBACK_MONTHS, 2),
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb29_summary_path = ARTIFACTS_DIR / "notebook_29_summary.json"
with open(nb29_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_29_summary, f, indent=2)
print(f"\u2705 Saved -> {nb29_summary_path.name} (this problem's artifacts folder)")
print("\n\u2705 Section 13 complete.")


# =============================================================================
# SECTION 14: COMPLETION SUMMARY -- PROBLEM 4 COMPLETE, NEXT: PROBLEM 3
# =============================================================================
_section("SECTION 14: Notebook 29 Complete -- Problem 4 Complete, Next: Problem 3")

print("NOTEBOOK 29: FINANCIAL-IMPACT REPORTING & PACKAGING -- COMPLETE")
print("PROBLEM 4 (DELINQUENCY ESCALATION / LOSS SEVERITY) -- ALL 4 NOTEBOOKS COMPLETE")
print(f"  Loss recognition change (flat vs tier LGD) : ${LOSS_DELTA_USD:,.0f}")
print(f"  Estimated loss prevented per cycle          : ${LOSS_PREVENTED_USD:,.0f}")
print(f"  Estimated Year-1 ROI / payback               : {ROI_PCT:,.0f}% / {PAYBACK_MONTHS:.1f} months")
print(f"  Files produced                              : {len(_expected_files) + 1}")
for _p in _expected_files + [nb29_summary_path]:
    print(f"    - {_p.name}")
print("  Next: Problem 3 (Expected Credit Loss / IFRS9-CECL) -- consumes this problem's tier-LGD output.")
print("\n\u2705 Ready to proceed.")
